In [1]:
!pip install tcav


In [2]:
!pip install tensorflow scikit-learn matplotlib


  Using cached numpy-1.23.5-cp311-cp311-win_amd64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-win_amd64.whl (14.6 MB)
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.0
    Uninstalling wrapt-1.17.0:
      Successfully uninstalled wrapt-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blis 1.0.1 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.23.5 which is incompatible.
geemap 0.35.1 requires folium>=0.17.0, but you have folium 0.14.0 which is incompatible.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 1.23.5 which is incompatible.
rasterio 1.4.3 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.0 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
thinc 8.3.2 requires numpy<2.1.0,>=2.0.0; python_version >= "3.9", but you have numpy 1.23.5 which is incompatible.
xarray 2024.11.0 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.linear_model import SGDClassifier
import matplotlib.pyplot as plt
import os

# Step 1: Load Pretrained Model
model = InceptionV3(weights='imagenet')
layer_name = 'mixed10'  # Layer to get activations from

intermediate_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer(layer_name).output
)

# Step 2: Load a local zebra image
img_path = r"C:\Users\Aneesh Mada\OneDrive\Pictures\Screenshots\Screenshot 2025-04-09 124640.png"  # Ensure this image exists in your working directory
img = load_img(img_path, target_size=(299, 299))
x = img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

preds = model.predict(x)
print("Prediction:", decode_predictions(preds, top=3)[0])

# Step 3: Generate synthetic "striped" concept images
def generate_striped_images(num=20):
    images = []
    for _ in range(num):
        img = np.zeros((299, 299, 3), dtype=np.uint8)
        for i in range(0, 299, 10):
            img[:, i:i+5, :] = 255  # white vertical stripes
        images.append(preprocess_input(np.expand_dims(img, axis=0)))
    return np.vstack(images)

striped_images = generate_striped_images(20)

# Step 4: Extract activations
concept_activations = intermediate_model.predict(striped_images)
image_activations = intermediate_model.predict(x)

# Step 5: Train CAV (Concept Activation Vector)
X = np.concatenate([concept_activations, np.random.normal(size=concept_activations.shape)])
y = np.array([1] * len(concept_activations) + [0] * len(concept_activations))

X_flat = X.reshape(X.shape[0], -1)
clf = SGDClassifier().fit(X_flat, y)
cav = clf.coef_.reshape(concept_activations.shape[1:])

# Step 6: Compute directional derivative
image_activ_flat = image_activations.reshape(-1)
cav_flat = cav.reshape(-1)
directional_derivative = np.dot(image_activ_flat, cav_flat)
print(f"Directional derivative: {directional_derivative:.4f}")

# Step 7: TCAV Score: high means concept strongly influences the prediction
tcav_score = np.mean([np.dot(act.reshape(-1), cav_flat) > 0 for act in concept_activations])
print(f"TCAV Score for 'striped' concept influencing 'zebra': {tcav_score:.2f}")

# Step 8: Visualize
plt.imshow(img)
plt.title("Target Image: Zebra")
plt.axis('off')
plt.show()

# Visualize a striped concept
plt.imshow(generate_striped_images(1)[0].astype(np.uint8))
plt.title("Concept Image: Striped")
plt.axis('off')
plt.show()



16162816/96112376 [====>.........................] - ETA: 4:02 

In [14]:
# Install dependencies
!pip install transformers tcav scikit-learn torch --quiet

# Import modules
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.linear_model import SGDClassifier

# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
model.eval()
print("Model loaded ✅")

# Define example texts
concept_texts = [
    "I am so happy today!",
    "What a wonderful surprise.",
    "Joy is in the air.",
    "I'm feeling great and excited!"
]

target_texts = [
    "The movie was fantastic.",
    "I love this product.",
    "She was amazing on stage.",
    "It was a pleasant experience."
]

random_texts = [
    "The book is on the table.",
    "He went to the store.",
    "It is raining outside.",
    "She has a blue bag."
]

# Function to extract CLS token embedding from BERT
def get_cls_embedding(texts, layer=-2):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    hidden_states = outputs.hidden_states
    cls_embeddings = hidden_states[layer][:, 0, :]  # CLS token
    return cls_embeddings.numpy()

# Get embeddings
concept_acts = get_cls_embedding(concept_texts)
target_acts = get_cls_embedding(target_texts)
random_acts = get_cls_embedding(random_texts)

# Train linear classifier to get CAV (Concept Activation Vector)
X = np.concatenate([concept_acts, random_acts])
y = np.concatenate([np.ones(len(concept_acts)), np.zeros(len(random_acts))])

clf = SGDClassifier(alpha=0.01, max_iter=1000)
clf.fit(X, y)

cav = clf.coef_

# Compute TCAV score
dot_products = np.dot(target_acts, cav.T)
tcav_score = np.mean(dot_products > 0)

print("\n✅ TCAV Score (alignment of positive sentiment with joy):", round(float(tcav_score), 3))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.5 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model loaded ✅

✅ TCAV Score (alignment of positive sentiment with joy): 1.0
